In [ ]:
# The transformers library is still necessary for the underlying BART model architecture.
from transformers import BartTokenizer, BartConfig, BartForConditionalGeneration

# Define the model name from the Hugging Face Hub.
# This tells the library where to download the necessary files from.
model_name = "Babelscape/rebel-large"

# --- Step 1: Manually load the Tokenizer, Configuration, and Model ---

# Load the specific BART tokenizer from the pre-trained model files.
print("Loading tokenizer...")
tokenizer = BartTokenizer.from_pretrained(model_name)

# Load the specific BART model configuration.
# This file contains all the model's architectural settings.
print("Loading model configuration...")
config = BartConfig.from_pretrained(model_name)

# Load the BART model for sequence-to-sequence tasks with the pre-trained weights.
print("Loading pre-trained model weights...")
model = BartForConditionalGeneration.from_pretrained(model_name, config=config)
print("Model loaded successfully.")

In [ ]:
# Use triple quotes for multi-line strings
text = """France,[h] officially the French Republic,[i] is a country primarily located in Western Europe. Its overseas regions and territories include French Guiana in South America, Saint Pierre and Miquelon in the North Atlantic, the French West Indies, and many islands in Oceania and the Indian Ocean, giving it the largest discontiguous exclusive economic zone in the world. Metropolitan France shares borders with Belgium and Luxembourg to the north; Germany to the northeast; Switzerland to the east; Italy and Monaco to the southeast; Andorra and Spain to the south; and a maritime border with the United Kingdom to the northwest. Its metropolitan area extends from the Rhine to the Atlantic Ocean and from the Mediterranean Sea to the English Channel and the North Sea. Its eighteen integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi) and have an estimated total population of over 68.6 million as of January 2025. France is a semi-presidential republic. Its capital, largest city and main cultural and economic centre is Paris.

Metropolitan France was settled during the Iron Age by Celtic tribes known as Gauls before Rome annexed the area in 51 BC, leading to a distinct Gallo-Roman culture. In the Early Middle Ages, the Franks formed the kingdom of Francia, which became the heartland of the Carolingian Empire. The Treaty of Verdun of 843 partitioned the empire, with West Francia evolving into the Kingdom of France. In the High Middle Ages, France was a powerful but decentralised feudal kingdom, but from the mid-14th to the mid-15th centuries, France was plunged into a dynastic conflict with England known as the Hundred Years' War. In the 16th century, French culture flourished during the French Renaissance and a French colonial empire emerged. Internally, France was dominated by the conflict with the House of Habsburg and the French Wars of Religion between Catholics and Huguenots. France was successful in the Thirty Years' War and further increased its influence during the reign of Louis XIV.

The French Revolution of 1789 overthrew the Ancien Régime and produced the Declaration of the Rights of Man, which expresses the nation's ideals to this day. France reached its political and military zenith in the early 19th century under Napoleon Bonaparte, subjugating part of continental Europe and establishing the First French Empire. The collapse of the empire initiated a period of relative decline, in which France endured the Bourbon Restoration until the founding of the French Second Republic which was succeeded by the Second French Empire upon Napoleon III's takeover. His empire collapsed during the Franco-Prussian War in 1870. This led to the establishment of the Third French Republic, and subsequent decades saw a period of economic prosperity and cultural and scientific flourishing known as the Belle Époque. France was one of the major participants of World War I, from which it emerged victorious at great human and economic cost. It was among the Allies of World War II, but it surrendered and was occupied in 1940. Following its liberation in 1944, the short-lived Fourth Republic was established and later dissolved in the course of the defeat in the Algerian War. The current Fifth Republic was formed in 1958 by Charles de Gaulle. Algeria and most French colonies became independent in the 1960s, with the majority retaining close economic and military ties with France.

France retains its centuries-long status as a global centre of art, science, and philosophy. It hosts the fourth-largest number of UNESCO World Heritage Sites and is the world's leading tourist destination, having received 100 million foreign visitors in 2023. A developed country, France has a high nominal per capita income globally, and its economy ranks among the largest in the world by both nominal GDP and PPP-adjusted GDP. It is a great power, being one of the five permanent members of the United Nations Security Council and an official nuclear-weapon state. The country is part of multiple international organisations and forums."""

# Tokenize the text
input_ids = tokenizer(text, return_tensors="pt").input_ids

# Generate the linearized triplets
generated_ids = model.generate(input_ids)

# Decode the output
decoded_output = tokenizer.decode(generated_ids[0], skip_special_tokens=False)

# Print the output to see the result
print(decoded_output)

In [ ]:
import csv

# Assume 'decoded_output' contains the raw string from the REBEL model.
# --- 1. Parsing Logic (from previous script) ---
# Clean up the start and end tokens from the raw output.
decoded_output = decoded_output.replace("<s>", "").replace("</s>", "")

triplets = []
# Split the string by the '<triplet>' token to process each group.
relation_groups = decoded_output.split('<triplet>')

current_head_entity = None
for group in relation_groups:
    if not group.strip():
        continue
        
    # Split the group into the head entity and the rest of the relations.
    parts = group.split('<subj>')
    head_entity = parts[0].strip()
    
    # Process each relation associated with the head entity.
    for relation_part in parts[1:]:
        try:
            tail_entity, relation = relation_part.split('<obj>')
            # Append the extracted triplet as a tuple.
            triplets.append((head_entity.strip(), relation.strip(), tail_entity.strip()))
        except ValueError:
            # Skip any malformed relation parts.
            continue

# --- 2. Save the Triplets to a CSV file ---
csv_file_name = "triplets.csv"

# 'with open' ensures the file is properly closed after writing.
with open(csv_file_name, mode='w', newline='', encoding='utf-8') as file:
    # Create a csv writer object.
    writer = csv.writer(file)
    
    # Write the header row.
    writer.writerow(['head', 'relation', 'tail'])
    
    # Write all the triplet data rows.
    writer.writerows(triplets)

print(f"✅ Successfully saved {len(triplets)} triplets to '{csv_file_name}'")

In [ ]:
import requests
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import re
import json
from typing import List, Dict, Tuple

def get_wikipedia_content(title: str) -> str:
    """Fetch Wikipedia page content via API with better error handling"""
    try:
        # Use the more reliable Wikipedia API endpoint
        wiki_url = "https://en.wikipedia.org/w/api.php"
        params = {
            'action': 'query',
            'format': 'json',
            'titles': title,
            'prop': 'extracts',
            'exintro': True,
            'explaintext': True,
            'exsectionformat': 'plain',
            'redirects': 1  # Follow redirects
        }
        
        response = requests.get(wiki_url, params=params, timeout=10)
        response.raise_for_status()  # Raise exception for bad status codes
        
        data = response.json()
        pages = data.get('query', {}).get('pages', {})
        
        # Get the first (and usually only) page
        page_id = list(pages.keys())[0]
        page_data = pages[page_id]
        
        if 'missing' in page_data:
            print(f"Wikipedia page '{title}' not found")
            return ""
        
        extract = page_data.get('extract', '')
        
        if not extract:
            print(f"No content found for '{title}'")
            return ""
            
        print(f"Successfully fetched Wikipedia content for '{title}'")
        return extract
        
    except requests.RequestException as e:
        print(f"Network error fetching Wikipedia: {e}")
        return ""
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        print(f"Response content: {response.text[:200]}...")
        return ""
    except Exception as e:
        print(f"Unexpected error: {e}")
        return ""

def get_fallback_france_text():
    """Fallback text about France if Wikipedia fails"""
    return """France,[h] officially the French Republic,[i] is a country primarily located in Western Europe. Its overseas regions and territories include French Guiana in South America, Saint Pierre and Miquelon in the North Atlantic, the French West Indies, and many islands in Oceania and the Indian Ocean, giving it the largest discontiguous exclusive economic zone in the world. Metropolitan France shares borders with Belgium and Luxembourg to the north; Germany to the northeast; Switzerland to the east; Italy and Monaco to the southeast; Andorra and Spain to the south; and a maritime border with the United Kingdom to the northwest. Its metropolitan area extends from the Rhine to the Atlantic Ocean and from the Mediterranean Sea to the English Channel and the North Sea. Its eighteen integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi) and have an estimated total population of over 68.6 million as of January 2025. France is a semi-presidential republic. Its capital, largest city and main cultural and economic centre is Paris.

Metropolitan France was settled during the Iron Age by Celtic tribes known as Gauls before Rome annexed the area in 51 BC, leading to a distinct Gallo-Roman culture. In the Early Middle Ages, the Franks formed the kingdom of Francia, which became the heartland of the Carolingian Empire. The Treaty of Verdun of 843 partitioned the empire, with West Francia evolving into the Kingdom of France. In the High Middle Ages, France was a powerful but decentralised feudal kingdom, but from the mid-14th to the mid-15th centuries, France was plunged into a dynastic conflict with England known as the Hundred Years' War. In the 16th century, French culture flourished during the French Renaissance and a French colonial empire emerged. Internally, France was dominated by the conflict with the House of Habsburg and the French Wars of Religion between Catholics and Huguenots. France was successful in the Thirty Years' War and further increased its influence during the reign of Louis XIV.

The French Revolution of 1789 overthrew the Ancien Régime and produced the Declaration of the Rights of Man, which expresses the nation's ideals to this day. France reached its political and military zenith in the early 19th century under Napoleon Bonaparte, subjugating part of continental Europe and establishing the First French Empire. The collapse of the empire initiated a period of relative decline, in which France endured the Bourbon Restoration until the founding of the French Second Republic which was succeeded by the Second French Empire upon Napoleon III's takeover. His empire collapsed during the Franco-Prussian War in 1870. This led to the establishment of the Third French Republic, and subsequent decades saw a period of economic prosperity and cultural and scientific flourishing known as the Belle Époque. France was one of the major participants of World War I, from which it emerged victorious at great human and economic cost. It was among the Allies of World War II, but it surrendered and was occupied in 1940. Following its liberation in 1944, the short-lived Fourth Republic was established and later dissolved in the course of the defeat in the Algerian War. The current Fifth Republic was formed in 1958 by Charles de Gaulle. Algeria and most French colonies became independent in the 1960s, with the majority retaining close economic and military ties with France.

France retains its centuries-long status as a global centre of art, science, and philosophy. It hosts the fourth-largest number of UNESCO World Heritage Sites and is the world's leading tourist destination, having received 100 million foreign visitors in 2023. A developed country, France has a high nominal per capita income globally, and its economy ranks among the largest in the world by both nominal GDP and PPP-adjusted GDP. It is a great power, being one of the five permanent members of the United Nations Security Council and an official nuclear-weapon state. The country is part of multiple international organisations and forums."""

def preprocess_text(text: str) -> List[str]:
    """Clean and split text into sentences"""
    if not text:
        return []
        
    # Remove citations like [1], [2], etc.
    text = re.sub(r'\[\d+\]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Better sentence splitting
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    
    # Filter out very short sentences and clean
    sentences = [s.strip() for s in sentences if len(s.strip()) > 30]
    
    return sentences[:5]  # Limit to first 5 sentences for testing

def parse_rebel_output(output_text: str) -> List[Tuple[str, str, str]]:
    """Parse REBEL output into (subject, relation, object) triples"""
    triples = []
    
    if not output_text or output_text.strip() == "":
        return triples
    
    # REBEL outputs triples in various formats, try to handle them all
    output_text = output_text.replace('<triplet>', '').replace('</triplet>', '')
    output_text = output_text.replace('<s>', '').replace('</s>', '')
    
    # Split by common delimiters
    parts = re.split(r'[;\n]', output_text)
    
    for part in parts:
        part = part.strip()
        if '|' in part:
            components = [c.strip() for c in part.split('|')]
            if len(components) >= 3:
                subject, relation, obj = components[0], components[1], components[2]
                if subject and relation and obj:  # All non-empty
                    triples.append((subject, relation, obj))
    
    return triples

def test_rebel_on_france():
    """Main function to test REBEL on France Wikipedia page"""
    
    print("Loading REBEL model (this may take a few minutes)...")
    try:
        # Load pre-trained REBEL model
        tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
        model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")
        print("✓ REBEL model loaded successfully")
    except Exception as e:
        print(f"✗ Error loading REBEL model: {e}")
        print("Make sure you have installed: pip install transformers torch")
        return
    
    print("\nFetching Wikipedia content for France...")
    france_text = get_wikipedia_content("France")
    
    if not france_text:
        print("Using fallback text about France...")
        france_text = get_fallback_france_text()
    
    print(f"Text length: {len(france_text)} characters")
    print(f"Preview: {france_text[:200]}...\n")
    
    # Preprocess text
    sentences = preprocess_text(france_text)
    print(f"Processing {len(sentences)} sentences...\n")
    
    all_triples = []
    
    for i, sentence in enumerate(sentences):
        print(f"Sentence {i+1}: {sentence[:80]}...")
        
        try:
            # Tokenize input
            inputs = tokenizer(sentence, 
                              max_length=512, 
                              padding=True, 
                              truncation=True, 
                              return_tensors="pt")
            
            # Generate triples
            outputs = model.generate(inputs["input_ids"], 
                                   max_length=256,
                                   num_beams=3,
                                   early_stopping=True,
                                   do_sample=False)
            
            # Decode output
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            print(f"REBEL output: {decoded}")
            
            # Parse triples
            triples = parse_rebel_output(decoded)
            
            if triples:
                print(f"✓ Extracted {len(triples)} triples:")
                for j, (subj, rel, obj) in enumerate(triples):
                    print(f"  {j+1}. [{subj}] --{rel}--> [{obj}]")
            else:
                print("  No triples extracted")
            
            all_triples.extend(triples)
            
        except Exception as e:
            print(f"✗ Error processing sentence: {e}")
            
        print("-" * 80)
    
    # Summary
    print(f"\n=== SUMMARY ===")
    print(f"Sentences processed: {len(sentences)}")
    print(f"Total triples extracted: {len(all_triples)}")
    
    if all_triples:
        # Show unique entities and relations
        entities = set()
        relations = set()
        
        for subj, rel, obj in all_triples:
            entities.add(subj)
            entities.add(obj)
            relations.add(rel)
        
        print(f"Unique entities: {len(entities)}")
        print(f"Unique relations: {len(relations)}")
        
        print(f"\nEntities found: {sorted(list(entities))[:10]}")
        print(f"Relations used: {sorted(list(relations))}")
        
        # Save results
        results = {
            'source': 'Wikipedia - France',
            'total_triples': len(all_triples),
            'triples': all_triples,
            'entities': sorted(list(entities)),
            'relations': sorted(list(relations))
        }
        
        try:
            with open('france_rebel_results.json', 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            print(f"\n✓ Results saved to 'france_rebel_results.json'")
        except Exception as e:
            print(f"✗ Error saving results: {e}")
    else:
        print("No triples were extracted. Try with different text or check model installation.")

if __name__ == "__main__":
    test_rebel_on_france()


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import re

def preprocess_text(text):
    # Remove citations like [1], etc.
    text = re.sub(r'\[\d+\]', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Simple sentence splitting (improve with spaCy/nltk if desired)
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [s.strip() for s in sentences if len(s.strip()) > 30]

def parse_rebel_output(output_text):
    triples = []
    output_text = output_text.replace('<triplet>', '').replace('</triplet>', '')
    parts = re.split(r'[;\n]', output_text)
    for part in parts:
        part = part.strip()
        if '|' in part:
            pieces = [x.strip() for x in part.split('|')]
            if len(pieces) == 3:
                triples.append(tuple(pieces))
    return triples

def run_rebel_on_text(raw_text):
    tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
    model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")
    sentences = preprocess_text(raw_text)
    all_triples = []
    for i, sentence in enumerate(sentences):
        inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512)
        outputs = model.generate(inputs["input_ids"], max_length=256, num_beams=3, early_stopping=True)
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        triples = parse_rebel_output(decoded)
        print(f"\nSentence {i+1}: {sentence[:80]}...")
        for subj, rel, obj in triples:
            print(f"  [{subj}] -- {rel} --> [{obj}]")
        all_triples.extend(triples)
    print(f"\nTotal triples extracted: {len(all_triples)}")

# Paste your Wikipedia content for France below
RAW_TEXT = """
France,[h] officially the French Republic,[i] is a country primarily located in Western Europe. Its overseas regions and territories include French Guiana in South America, Saint Pierre and Miquelon in the North Atlantic, the French West Indies, and many islands in Oceania and the Indian Ocean, giving it the largest discontiguous exclusive economic zone in the world. Metropolitan France shares borders with Belgium and Luxembourg to the north; Germany to the northeast; Switzerland to the east; Italy and Monaco to the southeast; Andorra and Spain to the south; and a maritime border with the United Kingdom to the northwest. Its metropolitan area extends from the Rhine to the Atlantic Ocean and from the Mediterranean Sea to the English Channel and the North Sea. Its eighteen integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi) and have an estimated total population of over 68.6 million as of January 2025. France is a semi-presidential republic. Its capital, largest city and main cultural and economic centre is Paris.

Metropolitan France was settled during the Iron Age by Celtic tribes known as Gauls before Rome annexed the area in 51 BC, leading to a distinct Gallo-Roman culture. In the Early Middle Ages, the Franks formed the kingdom of Francia, which became the heartland of the Carolingian Empire. The Treaty of Verdun of 843 partitioned the empire, with West Francia evolving into the Kingdom of France. In the High Middle Ages, France was a powerful but decentralised feudal kingdom, but from the mid-14th to the mid-15th centuries, France was plunged into a dynastic conflict with England known as the Hundred Years' War. In the 16th century, French culture flourished during the French Renaissance and a French colonial empire emerged. Internally, France was dominated by the conflict with the House of Habsburg and the French Wars of Religion between Catholics and Huguenots. France was successful in the Thirty Years' War and further increased its influence during the reign of Louis XIV.

The French Revolution of 1789 overthrew the Ancien Régime and produced the Declaration of the Rights of Man, which expresses the nation's ideals to this day. France reached its political and military zenith in the early 19th century under Napoleon Bonaparte, subjugating part of continental Europe and establishing the First French Empire. The collapse of the empire initiated a period of relative decline, in which France endured the Bourbon Restoration until the founding of the French Second Republic which was succeeded by the Second French Empire upon Napoleon III's takeover. His empire collapsed during the Franco-Prussian War in 1870. This led to the establishment of the Third French Republic, and subsequent decades saw a period of economic prosperity and cultural and scientific flourishing known as the Belle Époque. France was one of the major participants of World War I, from which it emerged victorious at great human and economic cost. It was among the Allies of World War II, but it surrendered and was occupied in 1940. Following its liberation in 1944, the short-lived Fourth Republic was established and later dissolved in the course of the defeat in the Algerian War. The current Fifth Republic was formed in 1958 by Charles de Gaulle. Algeria and most French colonies became independent in the 1960s, with the majority retaining close economic and military ties with France.

France retains its centuries-long status as a global centre of art, science, and philosophy. It hosts the fourth-largest number of UNESCO World Heritage Sites and is the world's leading tourist destination, having received 100 million foreign visitors in 2023. A developed country, France has a high nominal per capita income globally, and its economy ranks among the largest in the world by both nominal GDP and PPP-adjusted GDP. It is a great power, being one of the five permanent members of the United Nations Security Council and an official nuclear-weapon state. The country is part of multiple international organisations and forums.
"""

if __name__ == "__main__":
    run_rebel_on_text(RAW_TEXT)


In [ ]:
import os
import re
import google.generativeai as genai
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- Gemini API Configuration ---
# Configure the client library by setting the API key.
# It's recommended to set this as an environment variable for security.
try:
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
except KeyError:
    print("❗ GOOGLE_API_KEY environment variable not set.")
    print("Please set the GOOGLE_API_KEY environment variable to your API key.")
    exit()

def call_gemini_api(prompt: str) -> str:
    """
    Calls the Gemini API with the given prompt and returns the generated text.
    Uses the official Google AI Python SDK.
    """
    try:
        # Initialize the model. Using 'gemini-1.5-flash-latest' for speed and efficiency.
        model = genai.GenerativeModel('gemini-1.5-flash-latest')

        # Set generation configuration for controlling the output
        generation_config = genai.GenerationConfig(
            temperature=0.3,
            top_p=0.9,
            max_output_tokens=512,
            # stop_sequences=None # Can be added if needed
        )

        # Generate content
        response = model.generate_content(prompt, generation_config=generation_config)
        
        # Extract and return the text part of the response
        return response.text.strip()
    except Exception as e:
        print(f"An error occurred during the Gemini API call: {e}")
        return "" # Return an empty string on failure

def call_gemini_refiner(raw_text: str) -> str:
    """
    Constructs a specific prompt for text refinement and calls the Gemini API.
    """
    prompt = f"""
You are a text refiner specialized to help knowledge graph extraction. Your task is to rewrite the following input into clear, unambiguous, simple declarative sentences suitable for knowledge extraction models like REBEL.

- Break complex sentences into multiple simpler sentences if needed.
- Remove ambiguity, pronouns, and unnecessary references.
- Preserve all factual information as precisely as possible.
- Avoid introducing new information or changing facts.

Input text:
\"\"\"
{raw_text}
\"\"\"

Please output only the refined text. Keep the output concise and easy to parse.
"""
    return call_gemini_api(prompt)

def preprocess_text_for_rebel(text: str):
    """
    Prepares text for the REBEL model by cleaning and splitting it into sentences.
    """
    # Remove citations like [1], [23], etc.
    text = re.sub(r'\[\d+\]', '', text)
    # Collapse multiple whitespace characters into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    # A simple regex for sentence splitting. For more complex texts, libraries like spaCy or NLTK are better.
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    # Filter out very short strings that are unlikely to be full sentences
    return [s.strip() for s in sentences if len(s.strip()) > 30]

def parse_rebel_output(output_text: str):
    """
    Parses the REBEL model's specific output format into a list of (subject, relation, object) tuples.
    """
    triples = []
    # Clean the raw output string from the model
    clean_text = output_text.replace('<triplet>', '').replace('</triplet>', '').strip()
    
    # Split the string by newlines or semicolons which often delimit triples
    potential_triples = re.split(r'[;\n]', clean_text)
    
    for part in potential_triples:
        part = part.strip()
        if '|' in part:
            pieces = [p.strip() for p in part.split('|')]
            if len(pieces) == 3:
                # Unpack and append the triple
                subject, relation, obj = pieces
                triples.append((subject, relation, obj))
    return triples

def run_rebel_on_text(raw_text: str):
    """
    Main pipeline: Refines text with Gemini, then runs REBEL to extract knowledge triples.
    """
    # Load the REBEL model and tokenizer from Hugging Face
    tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
    model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")
    
    print("1. Refining input text with Gemini 1.5 Flash... 📝")
    refined_text = call_gemini_refiner(raw_text)
    if not refined_text:
        print("Text refinement failed. Exiting.")
        return

    print(f"\n--- Refined Text ---\n{refined_text}\n----------------------\n")
    
    sentences = preprocess_text_for_rebel(refined_text)
    all_triples = []
    
    print(f"2. Extracting triples with REBEL from {len(sentences)} sentences... 🧠")
    for i, sentence in enumerate(sentences):
        # Tokenize the sentence for the model
        inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512)
        
        # Generate the output from the model
        outputs = model.generate(
            inputs["input_ids"],
            max_length=256,
            num_beams=3,
            early_stopping=True
        )
        
        # Decode the generated tokens back into a string
        decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Raw REBEL output:\n{decoded_output}\n")  # See exactly what REBEL generated

        triples = parse_rebel_output(decoded_output)
        
        print(f"\nSentence {i+1}: '{sentence[:80]}...'")
        if triples:
            for subj, rel, obj in triples:
                print(f"  ✅ [{subj}] --({rel})--> [{obj}]")
            all_triples.extend(triples)
        else:
            print("  ❌ No triples extracted.")
            
    print(f"\n✨ Total triples extracted: {len(all_triples)}")

if __name__ == "__main__":
    # Example text about France. You can replace this with any text.
    RAW_TEXT = """
The Battle of Sinhagad, also known as Battle of Kondhana, involved an attack by Marathas during the night of 4 February 1670 on the Mughal fort of Sinhagad (then Kondhana), near the city of Pune, Maharashtra. The Marathas captured the fort.

Battle
Sinhagad was one of the first forts which Shivaji recaptured from the Mughals following the treaty of Purandar. The capture was made possible by scaling the walls at night with rope ladders. Tanaji Malusare was killed, but the fort was captured by the Marathas. The battle and Tanaji's exploits are the basis of a popular Marathi ballad.[2]

Aftermath
When Shivaji was informed about the victory and the loss of Tanaji's life during the battle, he is said to have exclaimed "गड आला पण सिंह गेला" (IAST: Gaḍa ālā paṇa siṃha gela English: The fort has been captured but we lost the lion). A bust of Tanaji Malusare was installed at the fort in memory of his contribution and sacrifice.[3]


"""
    run_rebel_on_text(RAW_TEXT)